In [ ]:
from matplotlib import pyplot as plt

x_list = [1e-6, 1e-5, 1e-4]
y_list = [2.5, 8, 16]

plt.figure(figsize=(8, 6))
plt.plot(x_list, y_list, 'o', color='black', markersize=10)
plt.xscale('log')
plt.xlabel('production rate (M/s)')
plt.ylabel('propagation rate (um/min)')
plt.show()

In [ ]:
# Read the file
with open('new_populated 15k.txt', 'r') as file:
    lines = file.readlines()

# Parse the data
data = []
for line in lines:
    if line.strip() and not line.startswith('//'):  # Skip empty lines and comments
        parts = line.strip().split(',')
        if len(parts) == 4:  # Ensure we have 4 parts (type, x, y, z)
            try:
                cell_type = parts[0]
                x = float(parts[1])
                y = float(parts[2])
                z = float(parts[3])
                data.append((cell_type, x, y, z))
            except ValueError:
                # Skip lines that can't be parsed
                continue

# Sort by sum of X and Y
sorted_by_sum = sorted(data, key=lambda entry: entry[1] + entry[2])

# Create the sorted output
sorted_output = []
for entry in sorted_by_sum:
    sorted_output.append(f"{entry[0]},{entry[1]},{entry[2]},{entry[3]}")

# Write to a new file
with open('new_populated_sum_sorted 15k.txt', 'w') as file:
    # Add the original comment header
    file.write("// filepath: /Users/huyunchen/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Melissa Kemp Lab/My Projects/ABM ROS/ROS-ABM/new_populated_sum_sorted.txt\n")
    file.write('\n'.join(sorted_output))

In [ ]:
import numpy as np
import random
from scipy.spatial import KDTree

def network_avg(positions):
    """Calculate the average center of all positions"""
    if not positions:
        return np.zeros(2)
    
    center = np.zeros(2)
    for x, y in positions:
        center += np.array([x, y])
    
    # Normalize
    center = center / len(positions)
    return center

def network_radius(positions):
    """Calculate the maximum distance from center to any position"""
    # First get the center
    center = network_avg(positions)
    
    # Get the maximum radial size of the network
    radius = -1
    for x, y in positions:
        r = np.array([x, y]) - center
        radius = max(np.linalg.norm(r), radius)
    
    return radius

def generate_populated_file(output_file, num_cells=8000, min_distance=13.0):
    """
    Generate a populated file with cells positioned at least min_distance apart
    All positions will be positive values.
    
    Args:
        output_file (str): Path to the output file
        num_cells (int): Number of cells to generate
        min_distance (float): Minimum distance between cell centers
    """
    # Define the bounds for x and y coordinates (all positive)
    # Making the area large enough to fit all cells with the minimum distance constraint
    side_length = np.sqrt(num_cells * min_distance * min_distance * 1.5)  # 1.5 is a factor to ensure enough space
    
    positions = []
    kdtree = None
    
    print(f"Generating {num_cells} cells with minimum distance {min_distance}...")
    print(f"Estimated area side length: {side_length}")
    
    # Generate first cell position randomly (all positive)
    x = random.uniform(0, side_length)
    y = random.uniform(0, side_length)
    positions.append((x, y))
    kdtree = KDTree(positions)
    
    attempts = 0
    max_attempts = 20000000  # Limit attempts to prevent infinite loop
    
    # Generate the rest of the positions
    while len(positions) < num_cells and attempts < max_attempts:
        attempts += 1
        
        # Generate a candidate position (all positive)
        x = random.uniform(0, side_length)
        y = random.uniform(0, side_length)
        
        # Check distance from existing positions
        if kdtree is not None:
            distances, _ = kdtree.query([(x, y)], k=1)
            if distances[0] >= min_distance:
                positions.append((x, y))
                # Update KDTree with new positions
                kdtree = KDTree(positions)
                
                # Print progress every 500 cells
                if len(positions) % 500 == 0:
                    print(f"Generated {len(positions)} cells...")
    
    if len(positions) < num_cells:
        print(f"Warning: Could only generate {len(positions)} cells out of {num_cells} requested")
    
    print(f"Writing to file {output_file}...")
    
    # Write positions to file
    with open(output_file, 'w') as file:
        for i, (x, y) in enumerate(positions):
            file.write(f"1,{x},{y},0.0\n")
    
    print(f"Successfully wrote {len(positions)} cells to {output_file}")
    print(f"Area dimensions: 0 to {side_length} for both x and y")

# Run the function with our parameters
output_file = "new_populated 15k.txt"
generate_populated_file(output_file, num_cells=15000, min_distance=13.0)

# Read the positions from the generated file
positions = []
with open(output_file, 'r') as file:
    for line in file:
        parts = line.strip().split(',')
        if len(parts) >= 3:  # Ensure we have at least 3 parts (id, x, y)
            x, y = float(parts[1]), float(parts[2])
            positions.append((x, y))

# Calculate and display network information
center = network_avg(positions)
radius = network_radius(positions)

print(f"Network center: ({center[0]:.2f}, {center[1]:.2f})")
print(f"Network radius: {radius:.2f}")


Generating 15000 cells with minimum distance 13.0...
Estimated area side length: 1950.0
Generated 500 cells...
Generated 1000 cells...
Generated 1500 cells...
Generated 2000 cells...
Generated 2500 cells...
Generated 3000 cells...
Generated 3500 cells...
Generated 4000 cells...
Generated 4500 cells...
Generated 5000 cells...
Generated 5500 cells...
Generated 6000 cells...
Generated 6500 cells...
Generated 7000 cells...
Generated 7500 cells...
Generated 8000 cells...
Generated 8500 cells...
Generated 9000 cells...
Generated 9500 cells...
Generated 10000 cells...
Generated 10500 cells...
Generated 11000 cells...
Generated 11500 cells...
Generated 12000 cells...
Generated 12500 cells...
Generated 13000 cells...
Generated 13500 cells...
Generated 14000 cells...
Generated 14500 cells...
Generated 15000 cells...
Writing to file new_populated 15k.txt...
Successfully wrote 15000 cells to new_populated 15k.txt
Area dimensions: 0 to 1950.0 for both x and y
Network center: (976.51, 975.93)
Networ

In [ ]:
import numpy as np
import os
import pickle

def export_locations(network_path, output_path, time_step=0):
    """
    Export cell locations to a txt file with 4 columns:
    Column 0: All 1s
    Columns 1-3: x, y, z coordinates from node.location
    
    Args:
        network_path: Path to the .gpickle file
        output_path: Directory to save the output txt file
        time_step: Time step for the network file (default 0)
    """
    # Create the full path to the gpickle file
    if network_path.endswith('.gpickle'):
        pickle_path = network_path
    else:
        pickle_path = f"{network_path}/network{time_step}.0.gpickle"
        
    # Load the network
    try:
        with open(pickle_path, 'rb') as f:
            network = pickle.load(f, encoding='latin1')
    except FileNotFoundError:
        print(f"File not found: {pickle_path}")
        return
    
    # Get the nodes
    nodes = network.nodes()
    
    # Create output filename based on input path
    base_name = os.path.basename(pickle_path).split('.')[0]
    output_file = os.path.join(output_path, f"{base_name}_locations.txt")
    
    # Write locations to file
    with open(output_file, 'w') as f:
        for node in nodes:
            x, y, z = node.location
            f.write(f"1,{x},{y},{z}\n")
    
    print(f"Exported {len(nodes)} cell locations to {output_file}")
    return output_file

path = '/Users/huyunchen/Documents/PhD/test/GSH, H2O2i with GJC, rand2/'
export_locations(path, path, time_step=50)

Exported 2131 cell locations to /Users/huyunchen/Documents/PhD/test/GSH, H2O2i with GJC, rand2/network50_locations.txt


'/Users/huyunchen/Documents/PhD/test/GSH, H2O2i with GJC, rand2/network50_locations.txt'